**Approach 2**

In this approach we will train a binary classification model for each genre.

**Set up:**
*   cloning git repository to utilize code we have there
*   mounting google drive (the free drive version space is not enough for this - 100 GB version is needed for the small dataset and 200GB for the medium dataset)
*   downloading and unzipping the Free Music Archive small dataset and metadata onto the drive

In [ ]:
!git clone https://github.com/majfu/MusicGenreClassifier.git

import sys
sys.path.append('/content/MusicGenreClassifier')

from src.config.config import *
from google.colab import drive

drive.mount(CONTENT_DRIVE_PATH)

import os
os.makedirs(os.path.dirname(PROJECT_FOLDER_PATH), exist_ok=True)

!wget -P {PROJECT_FOLDER_PATH} https://os.unil.cloud.switch.ch/fma/fma_metadata.zip
! unzip {METADATA_ZIP_FILE_PATH} -d {PROJECT_FOLDER_PATH}

!wget -P {PROJECT_FOLDER_PATH} https://os.unil.cloud.switch.ch/fma/fma_medium.zip
! unzip {FMA_MEDIUM_ZIP_PATH} -d {PROJECT_FOLDER_PATH}

In [ ]:
!pip install dotenv
!pip install scikit-multilearn
!pip install torchmetrics
!pip install pydub

**Creating labels file:**
*   Since this is a multilabel classification, the labels will be one-hot encoded
*   variable MIN_GENRE_SAMPLES_COUNT - genres with less than this many samples will not be included - the same setting as in approach 1
*   variable GENRES_TO_DROP - the same setting as in approach 1

In [ ]:
from src.data.label_encoder import LabelEncoder
from src.utils.labels_utils import downsample_labels_df
from src.utils.io_utils import *
from config.config import *

label_encoder = LabelEncoder()
one_hot_encoded_labels_df = label_encoder.get_one_hot_encoded_labels_df()

create_labels_file(one_hot_encoded_labels_df, INITIAL_ENCODED_LABELS_OUTPUT_PATH_2)

after that we are left with such statistics

genre counts:

| #   | Genre                    | Count |
|-----|---------------------------|-------|
| 0   | title_Rock                | 7103  |
| 1   | title_Electronic          | 6314  |
| 2   | title_Punk                | 3319  |
| 3   | title_Experimental        | 2251  |
| 4   | title_Hip-Hop             | 2201  |
| 5   | title_Pop                 | 1186  |
| 6   | title_Techno              | 802   |
| 7   | title_Classical           | 619   |
| 8   | title_Metal               | 577   |
| 9   | title_Old-Time / Historic | 510   |
| 10  | title_Dance               | 506   |

and the number of samples is 20184


**Handling .mp3 files:**
*   .mp3 format, the one that will be downloaded from FMA, is a compressed file version - we will convert them to .wav format; the function will return ids of files that could not be converted; the folder structure will not be preserved - it will be flattened; to save time, we will only convert the files which are in our labels file

In [ ]:
from src.utils.metadata_utils import get_track_ids_list

track_ids_list = get_track_ids_list(one_hot_encoded_labels_df)
corrupted_track_ids = convert_selected_mp3_to_wav(track_ids_list)

i got corrupted track ids: [1486, 5574, 65753, 98571, 98559, 98558, 98560, 99134, 105247, 108925, 127336, 133297, 143992]

**Updating labels file**
*   In the dataset most of the audio files are 30 seconds long, but there are outliers; finding and handling them is described in exploring_audio_legth_and_sampling_rate.ipynb, the ids are stored in LENGTH_OUTLIERS_TRACK_IDS variable (LENGTH_OUTLIERS_TRACK_IDS = [98569, 98567, 98568, 98566, 98565])
*   samples with ids which are in LENGTH_OUTLIERS_TRACK_IDS and corruptes_track_ids will be removed from the labels file
*   I will also remove .wav files which are in LENGTH_OUTLIERS_TRACK_IDS

In [ ]:
track_ids_to_remove = LENGTH_OUTLIERS_TRACK_IDS + corrupted_track_ids

one_hot_encoded_labels_df = one_hot_encoded_labels_df[~one_hot_encoded_labels_df['track_id'].isin(track_ids_to_remove)]

create_labels_file(one_hot_encoded_labels_df, ENCODED_LABELS_OUTPUT_PATH_2)

In [ ]:
remove_selected_wav_files(track_ids_to_remove)

**Creating datasets splits:**
*   Three splits will be created: training, validation and test for each genre
*   variables VAL_RATION and TEST_RATIO - they are set to 0.1 each


In [ ]:
from src.utils.labels_utils_approach2 import create_label_files_for_each_genre

create_label_files_for_each_genre(one_hot_encoded_labels_df)

**Extracting features:**
*   Spectrograms will be extracted beforehand to speed up the training runtime, we have enought space on the Google Drive to store the files
*   The feature vectors will be stored in tensors for compatibility with pytorch
*   Feature extraction for audio files is described more in depth in the audio-preprocessing-notes.pdf which is available in the git repository
*   MEL_BANDS_NUMBER = 60
*   thus extracted features are of shape 2998x60

In [ ]:
from src.features.feature_extractor import FeatureExtractor

feature_extractor = FeatureExtractor()
create_and_save_feature_arrays(feature_extractor)

The next step is to zip the spectrograms folder as well as genres label files folder and set their access to public.
The training scripts are saved in github in MGCmodel directory.